# Setup

In [ ]:
from gp import GaussianProcess
import pandas as pd
import jax
import jax.numpy as jnp
import jax.random as jr

from sklearn.preprocessing import MinMaxScaler
import wifiplotting as wp
from wifiplotting import *

from functools import partial

from time import time

In [ ]:
import matplotlib.pyplot as plt
# Configure font sizes.
TICK_SIZE = 6
SMALL_SIZE = 6
MEDIUM_SIZE = 8
BIGGER_SIZE = 11
plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=MEDIUM_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=TICK_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=TICK_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

MARKERSIZE = 3
LINEWIDTH = 0.5
plt.rc("lines", markersize=MARKERSIZE, linewidth=LINEWIDTH)
plt.rc("grid", linewidth=0.5)
plt.rc("grid", alpha=0.5)

# colorblind friendly colors: https://gist.github.com/thriveth/8560036
colors = ["#377eb8", "#ff7f00", "#4daf4a", "#f781bf", "#a65628", "#984ea3", "#999999", "#e41a1c", "#dede00"]


In [ ]:
osm_context = wp.OSMPlotContext.from_bounds(
    init_lons=[BR_CORNER[1], TL_CORNER[1]], init_lats=[BR_CORNER[0], TL_CORNER[0]],
    pad_fraction=0.0
)

In [ ]:
sequoia = pd.read_csv('../data/sequoia.csv')
sequoia.head()

In [ ]:
sequoia_nonan = sequoia[sequoia.rssi_signal_measured]

scaler = MinMaxScaler()
scaler.fit(pd.DataFrame(np.stack([TL_CORNER[::-1], BR_CORNER[::-1]]), columns=['longitude', 'latitude']))
# scaler.fit(sequoia_nonan[['longitude', 'latitude']])
X_train = pd.DataFrame()
X_train[['longitude', 'latitude']] = scaler.transform(sequoia_nonan[['longitude', 'latitude']])
X_train['indoor'] = sequoia_nonan['indoor'].astype(float).values

y_train = sequoia_nonan['rssi_sample']

X_train = jnp.array(X_train)
y_train = jnp.array(y_train)

wlon_train, wlat_train = osm_context.to_world(sequoia_nonan.longitude, sequoia_nonan.latitude)

Baseline: KNN
1. location only
2. group means, location kernel
3. group means, indoor/out kernel

In [ ]:
indoor_mask = X_train[:,2].astype(bool)
outdoor_mask = ~X_train[:,2].astype(bool)

stat = jnp.mean
stat1 = jnp.mean
stat2 = jnp.mean

def m1(obs):
    return y_train.mean()

def m2(obs):
    return jnp.mean(y_train[indoor_mask]) * obs[2] + jnp.mean(y_train[outdoor_mask]) * (1-obs[2])

    
def dist1(obs1, obs2):
    '''
    Location distance only
    '''
    return jnp.linalg.norm(obs1[:2] - obs2[:2], ord=2)**2


def dist3(obs1, obs2):
    '''
    Location + indoor/outdoor
    '''
    return jnp.linalg.norm(obs1 - obs2, ord=2)**2

scale = min([y_train[indoor_mask].var(), y_train[outdoor_mask].var()])
bandwidth = 1e-2
def K(obs1, obs2, dist):
    d = dist(obs1, obs2)
    return scale * jnp.exp(-d / (2*bandwidth))

K1 = partial(K, dist=dist1)
K3 = partial(K, dist=dist3)

# Convergence Diagnostics

In [ ]:
GP1 = GaussianProcess(m1, K1)
GP1.fit(X_train, y_train)

In [ ]:
start = time()
chain = GP1.gibbs(chains=10, samples=100)
end = time()
print(end - start)

In [ ]:
ind = jnp.where(X_train[:,2] == 1)[0][0]
outd= jnp.where(X_train[:,2] == 0)[0][0]

In [ ]:
covs = jnp.concatenate(chain[1])
post_variances = covs[:,[ind, outd]]

In [ ]:
fig = plt.figure(figsize=(4, 3), dpi=300)
plt.plot(range(post_variances.shape[0]), post_variances[:,0], label=r'$\sigma_{in}$')
plt.plot(range(post_variances.shape[0]), post_variances[:,1], label=r'$\sigma_{out}$')
plt.legend()
plt.xlabel('Gibbs Step')
plt.title('Gibbs Trace Plot for Posterior Variances')
# plt.savefig('../plots/variance_traceplots.pdf', bbox_inches='tight')

In [ ]:
def gelman_rubin_stat(chains):
    """
    chains: np.array of shape (m_chains, n_samples, d_dimensions)
    returns: np.array of shape (d_dimensions,) containing R-hat for each parameter
    """
    m, n, d = chains.shape
    
    if m < 2:
        raise ValueError("R-hat requires at least two chains.")

    # 1. Within-chain variance (W)
    # Shape: (m, d) -> Then mean across chains (axis=0) -> (d,)
    chain_vars = jnp.var(chains, axis=1, ddof=1)
    W = jnp.mean(chain_vars, axis=0)
    
    # 2. Between-chain variance (B)
    # Shape of means: (m, d)
    chain_means = jnp.mean(chains, axis=1)
    overall_means = jnp.mean(chain_means, axis=0)
    B = (n / (m - 1)) * jnp.sum((chain_means - overall_means)**2, axis=0)
    
    # 3. Estimated marginal posterior variance (V_hat)
    # Shape: (d,)
    v_hat = ((n - 1) / n) * W + (B / n)
    
    # 4. R-hat statistic
    # Shape: (d,)
    r_hat = jnp.sqrt(v_hat / W)
    return r_hat


In [ ]:
r_hat = gelman_rubin_stat(chain[0])

In [ ]:
r_hat.max()

# Visualizations

In [ ]:
refit = True
if refit:
    GP1 = GaussianProcess(m1, K1)
    GP2 = GaussianProcess(m2, K1)
    GP3 = GaussianProcess(m2, K3)
    
    GPs = [GP1, GP2, GP3]
    chains = [None] * 3
    
    key = jr.PRNGKey(305)
    for i, GP in enumerate(GPs):
        GP.fit(X_train, y_train)
        chains[i] = GP.gibbs(chains=4, samples=100)

In [ ]:
grid_width = 100

x_new = jnp.linspace(0, 1, grid_width)
y_new = jnp.linspace(0, 1, grid_width)
grid_points = jnp.stack(jnp.meshgrid(x_new, y_new), axis=-1).reshape(-1, 2)

coord_grid = scaler.inverse_transform(grid_points)
long_new, lat_new = coord_grid.T
wlon_new, wlat_new = osm_context.to_world(long_new, lat_new)

z_new = osm_context.contains_building(long_new, lat_new).reshape(-1,1)

X_new = jnp.concatenate([grid_points, z_new], axis=-1)

In [ ]:
y_means = [None] * 3
for i, GP in enumerate(GPs):
    y_means[i] = GP.posterior(X_new, chains[i][1])[0].mean(axis=0)

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [ ]:
vmax = y_train.max() + 5
vmin = y_train.min() - 5

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(6, 2), dpi=300)
for i in range(3):
    base_fig, base_ax, osm_metadata = osm_context.generate_base_axis(ax=axes[i],
            draw_buildings=False)
    
    axes[i].scatter(wlon_new, wlat_new, c=y_means[i], cmap="RdYlGn", vmin=vmin, vmax=vmax)
    
    sc = axes[i].scatter(wlon_train, wlat_train, c=y_train, cmap="RdYlGn",
                    vmin=vmin, vmax=vmax, s=1, alpha=0.4)
    plt.ticklabel_format(style='plain', axis='both', useOffset=False)

fig.suptitle('Learned Gaussian Process Posterior Mean Heatmaps')
axes[0].set_xlabel("Location Only")
axes[1].set_xlabel(r"Indoor-Aware $m$")
axes[2].set_xlabel(r"Indoor-Aware $m$ and $\mathcal{K}$")

divider = make_axes_locatable(axes[2])
cax = divider.append_axes("right", size="5%", pad=0.05)
fig.colorbar(axes[0].collections[0], cax, shrink=1)

plt.savefig('../plots/gp_heatmaps.pdf', bbox_inches='tight')